In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
sales_df=spark.read.csv(path="dbfs:/FileStore/tables/large_sales_data.csv",inferSchema=True,encoding="utf-8", header=True)

- check the schema

In [0]:
sales_df.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: integer (nullable = true)



- see the sample data


In [0]:
sales_df.show(10)

+--------------+----------+----------+----------------+--------+------+--------+
|transaction_id|      date|product_id|product_category|quantity| price|store_id|
+--------------+----------+----------+----------------+--------+------+--------+
|             1|2023-05-28|       131|     Electronics|       7|844.75|       3|
|             2|2023-11-08|       128|        Clothing|      10|105.23|       1|
|             3|2023-10-05|       136|        Clothing|       9|750.61|       5|
|             4|2023-05-17|       102|            Toys|       1|718.51|       1|
|             5|2023-06-10|       123|           Books|       4|386.34|       4|
|             6|2023-04-21|       198|     Electronics|       7|510.37|       4|
|             7|2023-03-03|       127|     Electronics|      16| 64.74|       5|
|             8|2023-01-16|       146|       Furniture|      18|281.08|       5|
|             9|2023-10-06|       176|            Toys|      15|920.89|       3|
|            10|2023-01-21| 

use display to see the data

In [0]:
display(sales_df.head(20))

transaction_id,date,product_id,product_category,quantity,price,store_id
1,2023-05-28,131,Electronics,7,844.75,3
2,2023-11-08,128,Clothing,10,105.23,1
3,2023-10-05,136,Clothing,9,750.61,5
4,2023-05-17,102,Toys,1,718.51,1
5,2023-06-10,123,Books,4,386.34,4
6,2023-04-21,198,Electronics,7,510.37,4
7,2023-03-03,127,Electronics,16,64.74,5
8,2023-01-16,146,Furniture,18,281.08,5
9,2023-10-06,176,Toys,15,920.89,3
10,2023-01-21,148,Furniture,17,779.99,1


count the row and columns

In [0]:
sales_df.columns

Out[6]: ['transaction_id',
 'date',
 'product_id',
 'product_category',
 'quantity',
 'price',
 'store_id']

In [0]:
len(sales_df.columns)

Out[7]: 7

In [0]:
sales_df.count()

Out[8]: 10000

count null values


In [0]:
is_null_count_check=sales_df.select([F.sum(F.col(col_name).isNull().cast("int")).alias("null_count_"+col_name) for col_name in sales_df.columns])
display(is_null_count_check)

null_count_transaction_id,null_count_date,null_count_product_id,null_count_product_category,null_count_quantity,null_count_price,null_count_store_id
0,0,0,0,0,0,0


- get summary

In [0]:
sales_df.na.df.summary()

Out[10]: DataFrame[summary: string, transaction_id: string, product_id: string, product_category: string, quantity: string, price: string, store_id: string]

In [0]:
display(sales_df.head(10))

transaction_id,date,product_id,product_category,quantity,price,store_id
1,2023-05-28,131,Electronics,7,844.75,3
2,2023-11-08,128,Clothing,10,105.23,1
3,2023-10-05,136,Clothing,9,750.61,5
4,2023-05-17,102,Toys,1,718.51,1
5,2023-06-10,123,Books,4,386.34,4
6,2023-04-21,198,Electronics,7,510.37,4
7,2023-03-03,127,Electronics,16,64.74,5
8,2023-01-16,146,Furniture,18,281.08,5
9,2023-10-06,176,Toys,15,920.89,3
10,2023-01-21,148,Furniture,17,779.99,1


# Basic Aggregations
1. Total Sales Amount Per Store:

  - Calculate the total sales amount (quantity * price) for each store.


In [0]:
total_sales_per_store=sales_df.groupBy("store_id").agg(F.round(F.sum(F.expr("quantity*price")),2).alias("total_sales")).orderBy(F.col("total_sales").desc())
display(total_sales_per_store)

store_id,total_sales
1,1.045000493E7
4,1.043893188E7
5,1.037585542E7
3,1.009676913E7
2,1.00857161E7


Databricks visualization. Run in Databricks to view.

2. Average Product Price by Category:

  - Find the average price of products within each product category.


In [0]:
average_product_price_per_product_category=sales_df.groupBy("product_category").agg(F.round(F.avg("price"),2).alias("average_product_price"))\
  .orderBy("average_product_price")
display(average_product_price_per_product_category)

product_category,average_product_price
Clothing,502.65
Electronics,506.8
Books,510.96
Furniture,511.87
Toys,515.15


Databricks visualization. Run in Databricks to view.

3. Total Quantity Sold Per Product:

  - Calculate the total quantity sold for each product.


In [0]:
display(sales_df.head(10))

transaction_id,date,product_id,product_category,quantity,price,store_id
1,2023-05-28,131,Electronics,7,844.75,3
2,2023-11-08,128,Clothing,10,105.23,1
3,2023-10-05,136,Clothing,9,750.61,5
4,2023-05-17,102,Toys,1,718.51,1
5,2023-06-10,123,Books,4,386.34,4
6,2023-04-21,198,Electronics,7,510.37,4
7,2023-03-03,127,Electronics,16,64.74,5
8,2023-01-16,146,Furniture,18,281.08,5
9,2023-10-06,176,Toys,15,920.89,3
10,2023-01-21,148,Furniture,17,779.99,1


In [0]:
quantity_sold_per_product=sales_df.groupBy("product_id").agg(F.sum("quantity").alias("total_quantity_sold"))
display(quantity_sold_per_product)

product_id,total_quantity_sold
148,1009
137,1074
133,953
108,1038
155,1086
193,978
126,1108
115,1133
101,835
183,968


Databricks visualization. Run in Databricks to view.

4. Daily Sales Amount:

  - Calculate the total sales amount for each day.

In [0]:
sales_per_day=sales_df.groupBy("date").agg(F.round(F.sum(F.expr("quantity*price")),2).alias("sales")).orderBy("date")
display(sales_per_day)

date,sales
2023-01-01,117368.16
2023-01-02,102408.64
2023-01-03,167807.89
2023-01-04,96865.05
2023-01-05,93265.09
2023-01-06,207569.35
2023-01-07,139103.04
2023-01-08,172959.09
2023-01-09,177814.85
2023-01-10,166080.67


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

# Grouping and Aggregation
5. Total Sales by Product Category:

  - Group the data by product category and calculate the total sales amount for each category.

In [0]:
sales_per_product_category=sales_df.groupBy("product_category").agg(F.round(F.sum(F.expr("quantity*price")),2).alias("sales"))
display(sales_per_product_category)

product_category,sales
Electronics,1.032899612E7
Clothing,1.017925016E7
Books,1.041975222E7
Furniture,1.008862907E7
Toys,1.043064989E7


Databricks visualization. Run in Databricks to view.

6. Store with Maximum Sales:

  - Identify the store that has the highest total sales.

In [0]:
store_wise_sales=sales_df.groupBy("store_id").agg(F.round(F.sum(F.expr("quantity*price")),2).alias("sales"))
highest_sales_store=store_wise_sales.select("*").orderBy(F.col("sales").desc()).first().asDict()
display(highest_sales_store)


{'store_id': 1, 'sales': 10450004.93}

7. Product with the Highest Total Revenue:

  - Identify the product that generated the most revenue (quantity * price).

In [0]:
revenue_per_product=sales_df.groupBy("product_id").agg(F.round(F.sum(F.expr("quantity*price")),2).alias("sales"))
heighest_revenue_product=revenue_per_product.select("sales","product_id").orderBy(F.col("sales").desc()).first().asDict()
print(heighest_revenue_product)


{'sales': 659529.6, 'product_id': 134}


8. Average Quantity Sold Per Day:

  - Calculate the average quantity sold per day across all stores.

In [0]:
average_quantity_sold_per_day=sales_df.groupBy(["date"]).agg(F.round(F.avg("quantity"),2).alias("quantity_sold"))
display(average_quantity_sold_per_day)

date,quantity_sold
2023-07-15,11.0
2023-06-22,11.36
2023-11-08,8.65
2023-05-22,8.85
2023-09-14,11.23
2023-06-18,10.69
2023-02-25,11.16
2023-11-22,9.19
2023-09-19,10.84
2023-06-23,9.89


# Complex Aggregations
9. Top 5 Best-Selling Products:

  - Find the top 5 products by total quantity sold.

In [0]:
quantity_sold_per_product=sales_df.groupBy("product_id").agg(F.sum("quantity").alias("quantity_sold")).orderBy(F.col("quantity_sold").desc())
top_5_product_quantity_sold=quantity_sold_per_product.limit(5)
display(top_5_product_quantity_sold)


product_id,quantity_sold
157,1276
110,1258
109,1198
118,1192
171,1184


Databricks visualization. Run in Databricks to view.

10. Sales Distribution by Store:

  - Calculate the percentage of total sales contributed by each store.

In [0]:
total_sales_per_store=sales_df.groupBy("store_id").agg(F.sum(F.expr("quantity*price")).alias("sales"))
total_sales=total_sales_per_store.agg(F.round(F.sum("sales"),2).alias("total_sales")).first()["total_sales"]
percentage_sales_contribution_par_store=total_sales_per_store.withColumn("percentage-sales-contribution",F.round((F.col("sales")/total_sales)*100,2))
display(percentage_sales_contribution_par_store)

store_id,sales,percentage-sales-contribution
1,1.0450004929999987E7,20.31
3,1.0096769130000025E7,19.63
5,1.0375855419999998E7,20.17
4,1.0438931879999995E7,20.29
2,1.0085716099999996E7,19.6


Databricks visualization. Run in Databricks to view.

11. Sales Trend Over Time:

  - Calculate the monthly sales trend for each product category.

In [0]:
monthly_sales_per_product_category=sales_df.withColumn("year_month", F.date_format(F.col("date"),"yyyy-MM")).groupBy(["year_month","product_category"]).agg(F.round(F.sum(F.expr("quantity*price")),2).alias("total_sale"))
display(monthly_sales_per_product_category)


year_month,product_category,total_sale
2023-01,Clothing,904839.6
2023-07,Books,965973.74
2023-03,Furniture,844349.96
2023-05,Clothing,874130.87
2023-12,Electronics,1019290.7
2023-02,Furniture,836674.22
2023-02,Books,853027.76
2023-02,Toys,816185.54
2023-12,Toys,794694.6
2023-09,Books,827725.2


Databricks visualization. Run in Databricks to view.

12. Most Popular Product Category by Store:

  - Identify the most popular product category in each store based on total quantity sold.

In [0]:
most_popular_products_by_store = sales_df.groupBy(["store_id","product_category"]).agg(F.sum("quantity").alias("quantity_sold")).withColumn("row_number", F.row_number().over(window=Window.partitionBy("store_id").orderBy(F.col("quantity_sold").desc()))).select("*").where(F.col("row_number")==1)


display(most_popular_products_by_store)

store_id,product_category,quantity_sold,row_number
1,Toys,4238,1
2,Clothing,4396,1
3,Clothing,4161,1
4,Books,4298,1
5,Electronics,4175,1


Databricks visualization. Run in Databricks to view.

13. Product Category with Highest Average Revenue:

  - Find the product category that has the highest average revenue per transaction

In [0]:
heighest_average_revenue_product_category=sales_df.groupBy("product_category").agg(F.round(F.avg(F.expr("quantity*price")),2).alias("avg_sales")).orderBy(F.col("avg_sales").desc())
print("heighest_averge_revenue_product_category::",heighest_average_revenue_product_category.first().asDict()["product_category"])
display(heighest_average_revenue_product_category)

heighest_averge_revenue_product_category:: Toys


product_category,avg_sales
Toys,5246.81
Books,5207.27
Clothing,5169.76
Furniture,5090.13
Electronics,5014.08


Databricks visualization. Run in Databricks to view.

# Window Functions
14. Cumulative Sales Per Store:

    -   Calculate the cumulative sales amount for each store over time.
